### делаем 2 коллекции

In [10]:
from platform import python_version
import os
import re
import uuid
import json
from collections import defaultdict

import pandas as pd
from dotenv import load_dotenv

from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

import clickhouse_connect

load_dotenv()



True

In [11]:
# ---------------------------------------------------
# 1. Подключения
# ---------------------------------------------------

QDRANT_URL = os.getenv("QDRANT_URL")
client_qdrant = QdrantClient(url=QDRANT_URL)

CH_HOST = os.getenv("CH_HOST", "84.201.160.255")
CH_PORT = int(os.getenv("CH_PORT", "8123"))
CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER", "peter")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "1234")

client_clickhouse = clickhouse_connect.get_client(
    host=CH_HOST,
    port=CH_PORT,
    username=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD
)

BASE_URL = os.getenv("EMBEDDINGS_BASE_URL", "http://localhost:8000/v1")

embeddings = OpenAIEmbeddings(
    model="Qwen/Qwen3-Embedding-0.6B",
    api_key="not-needed",
    base_url=BASE_URL,
    tiktoken_enabled=False,
)

vec = embeddings.embed_query("test")
EMBEDDING_DIM = len(vec)

print("Python:", python_version())
print("Embedding dim:", EMBEDDING_DIM)



Python: 3.10.18
Embedding dim: 1024


In [12]:
# ---------------------------------------------------
# 2. Имена коллекций
# ---------------------------------------------------

MESSAGES_COLLECTION = "mailkb_messages"
THREADS_COLLECTION = "mailkb_threads"


# ---------------------------------------------------
# 3. Настройка коллекций
# ---------------------------------------------------

def ensure_collection(collection_name: str, recreate: bool = False) -> QdrantVectorStore:
    collections = client_qdrant.get_collections().collections
    existing_names = {c.name for c in collections}

    if recreate and collection_name in existing_names:
        client_qdrant.delete_collection(collection_name=collection_name)
        existing_names.remove(collection_name)

    if collection_name not in existing_names:
        client_qdrant.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(
                size=EMBEDDING_DIM,
                distance=Distance.COSINE,
            ),
        )

    return QdrantVectorStore(
        client=client_qdrant,
        collection_name=collection_name,
        embedding=embeddings,
    )


# recreate=True только если хочешь пересобрать с нуля
messages_qv = ensure_collection(MESSAGES_COLLECTION, recreate=False)
threads_qv = ensure_collection(THREADS_COLLECTION, recreate=False)



In [13]:
# ---------------------------------------------------
# 4. Утилиты
# ---------------------------------------------------

RE_PREFIX = re.compile(r'^\s*(re|fw|fwd|aw|ответ):\s*', flags=re.IGNORECASE)


def normalize_subject(subj: str) -> str:
    s = (subj or "").strip()
    while True:
        ns = RE_PREFIX.sub('', s).strip()
        if ns == s:
            break
        s = ns
    s = re.sub(r'\s+', ' ', s)
    return s.lower()


def split_addrs(x) -> list[str]:
    if x is None:
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    return [p.strip() for p in re.split(r'[;,]', str(x)) if p.strip()]


def participants_list(row) -> list[str]:
    people = (
            split_addrs(row.get("from_addr")) +
            split_addrs(row.get("to_addr")) +
            split_addrs(row.get("cc_addr")) +
            split_addrs(row.get("bcc_addr"))
    )
    return sorted(set(people))


def safe_json_loads(value):
    if value is None:
        return None
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        try:
            return json.loads(value)
        except Exception:
            return None
    return None



In [14]:

# ---------------------------------------------------
# 5. SQL: берём только распарсенные письма
# ---------------------------------------------------

def load_join_batch(limit: int, offset: int) -> pd.DataFrame:
    query = f"""
    SELECT
        e.id,
        e.thread_key,
        e.message_id,
        e.subject,
        e.from_addr,
        e.to_addr,
        e.cc_addr,
        e.bcc_addr,
        e.sent_at_utc,
        e.folder,
        p.parsed_json
    FROM mailkb.emails e
    INNER JOIN mailkb.mail_parsed p
        ON e.id = p.email_id
    ORDER BY e.sent_at_utc ASC, e.id ASC
    LIMIT {limit} OFFSET {offset}
    """
    return client_clickhouse.query_df(query)




In [15]:
# ---------------------------------------------------
# 6. Построение документов для collection messages
# ---------------------------------------------------

def build_message_docs(df: pd.DataFrame) -> list[Document]:
    docs: list[Document] = []

    for row in df.to_dict("records"):
        parsed = safe_json_loads(row.get("parsed_json"))
        if not parsed:
            continue

        parsed_emails = parsed.get("emails", [])
        if not isinstance(parsed_emails, list):
            continue

        subj = (row.get("subject") or "").strip()
        norm_subj = normalize_subject(subj)
        participants = participants_list(row)

        for msg in parsed_emails:
            if not isinstance(msg, dict):
                continue

            email_body = (msg.get("email_body") or "").strip()
            if not email_body:
                continue

            topic = (msg.get("topic") or subj or "").strip()
            msg_date = msg.get("date")
            mail_query_number = msg.get("mail_query_number")
            keywords = msg.get("key_words") or []

            # thread_key пока MVP:
            # normalized subject
            thread_key = row.get("thread_key") or normalize_subject(subj)

            meta = {
                "email_id": row.get("id"),
                "message_id": row.get("message_id"),
                "subject": subj,
                "topic": topic,
                "date": str(msg_date) if msg_date is not None else str(row.get("sent_at_utc")),
                "mail_query_number": int(mail_query_number) if str(mail_query_number).isdigit() else mail_query_number,
                "keywords": keywords,
                "sent_at_utc": str(row.get("sent_at_utc")),
                "folder": row.get("folder"),
                "participants": participants,
                "thread_key": thread_key,
                "source_type": "message",
            }

            docs.append(
                Document(
                    page_content=email_body,
                    metadata=meta
                )
            )

    return docs


# ---------------------------------------------------
# 7. Стабильные id для messages
# ---------------------------------------------------

def make_message_ids(docs: list[Document]) -> list[str]:
    ids = []
    for d in docs:
        raw = (
            f"{d.metadata.get('email_id')}::"
            f"{d.metadata.get('mail_query_number')}::"
            f"{d.metadata.get('date')}::"
            f"{d.page_content[:100]}"
        )
        uid = uuid.uuid5(uuid.NAMESPACE_URL, raw)
        ids.append(str(uid))
    return ids




In [16]:
# ---------------------------------------------------
# 8. Загрузка messages в Qdrant
# ---------------------------------------------------

def upload_message_docs(docs: list[Document], qv: QdrantVectorStore):
    if not docs:
        return 0

    ids = make_message_ids(docs)
    texts = [d.page_content for d in docs]
    metadatas = [d.metadata for d in docs]

    qv.add_texts(
        texts=texts,
        metadatas=metadatas,
        ids=ids
    )
    return len(docs)


# ---------------------------------------------------
# 9. Полная загрузка messages
# ---------------------------------------------------

def ingest_messages(batch_size: int = 1000):
    offset = 0
    total_docs = 0
    total_rows = 0

    while True:
        df_batch = load_join_batch(limit=batch_size, offset=offset)
        if df_batch.empty:
            break

        docs = build_message_docs(df_batch)
        inserted = upload_message_docs(docs, messages_qv)

        total_docs += inserted
        total_rows += len(df_batch)

        print(
            f"[messages] rows_processed={total_rows}, "
            f"docs_inserted_total={total_docs}, "
            f"last_batch_rows={len(df_batch)}, "
            f"last_batch_docs={inserted}"
        )

        offset += len(df_batch)

    print(f"[messages] DONE: rows_processed={total_rows}, docs_inserted_total={total_docs}")


# ---------------------------------------------------
# 10. Загрузка данных для thread aggregation
# ---------------------------------------------------

def load_all_joined_df() -> pd.DataFrame:
    query = """
            SELECT e.id,
                   e.message_id,
                   e.subject,
                   e.from_addr,
                   e.to_addr,
                   e.cc_addr,
                   e.bcc_addr,
                   e.sent_at_utc,
                   e.folder,
                   p.parsed_json
            FROM mailkb.emails e
                     INNER JOIN mailkb.mail_parsed p
                                ON e.id = p.email_id
            ORDER BY e.sent_at_utc ASC, e.id ASC \
            """
    return client_clickhouse.query_df(query)


# ---------------------------------------------------
# 11. Агрегация threads
# ---------------------------------------------------

THREAD_SPLITTER = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150
)


def build_thread_docs(df: pd.DataFrame) -> list[Document]:
    grouped = defaultdict(list)

    for row in df.to_dict("records"):
        parsed = safe_json_loads(row.get("parsed_json"))
        if not parsed:
            continue

        parsed_emails = parsed.get("emails", [])
        if not isinstance(parsed_emails, list) or not parsed_emails:
            continue

        subj = (row.get("subject") or "").strip()
        thread_key = row.get("thread_key") or normalize_subject(subj)
        participants = participants_list(row)

        for msg in parsed_emails:
            if not isinstance(msg, dict):
                continue
            body = (msg.get("email_body") or "").strip()
            if not body:
                continue

            grouped[thread_key].append({
                "email_id": row.get("id"),
                "message_id": row.get("message_id"),
                "subject": subj,
                "topic": (msg.get("topic") or subj or "").strip(),
                "date": str(msg.get("date") or row.get("sent_at_utc")),
                "mail_query_number": msg.get("mail_query_number"),
                "keywords": msg.get("key_words") or [],
                "participants": participants,
                "body": body,
                "sent_at_utc": str(row.get("sent_at_utc")),
                "folder": row.get("folder"),
            })

    docs: list[Document] = []

    for thread_key, items in grouped.items():
        items_sorted = sorted(
            items,
            key=lambda x: (
                x.get("date") or "",
                str(x.get("mail_query_number") or "")
            )
        )

        subject = None
        participants = set()
        keywords = set()
        dates = []

        text_parts = []
        for item in items_sorted:
            if not subject and item.get("subject"):
                subject = item["subject"]

            for p in item.get("participants", []):
                participants.add(p)

            for kw in item.get("keywords", []):
                if kw:
                    keywords.add(str(kw))

            if item.get("date"):
                dates.append(item["date"])

            text_parts.append(
                "\n".join([
                    f"Topic: {item.get('topic') or ''}",
                    f"Date: {item.get('date') or ''}",
                    f"Message #{item.get('mail_query_number') or ''}",
                    item.get("body") or ""
                ]).strip()
            )

        full_text = "\n\n---\n\n".join(text_parts).strip()
        if not full_text:
            continue

        base_meta = {
            "thread_key": thread_key,
            "subject": subject or "",
            "participants": sorted(participants),
            "keywords": sorted(keywords),
            "message_count": len(items_sorted),
            "first_date": min(dates) if dates else None,
            "last_date": max(dates) if dates else None,
            "source_type": "thread",
        }

        # для длинных thread делаем chunking
        splitted = THREAD_SPLITTER.split_documents([
            Document(page_content=full_text, metadata=base_meta)
        ])
        docs.extend(splitted)

    return docs


# ---------------------------------------------------
# 12. Стабильные id для threads
# ---------------------------------------------------

def make_thread_ids(docs: list[Document]) -> list[str]:
    counters = {}
    ids = []

    for d in docs:
        thread_key = d.metadata.get("thread_key") or "no_thread"
        idx = counters.get(thread_key, 0)
        counters[thread_key] = idx + 1

        raw = f"{thread_key}::chunk_{idx}"
        uid = uuid.uuid5(uuid.NAMESPACE_URL, raw)
        ids.append(str(uid))

    return ids




In [17]:
# ---------------------------------------------------
# 13. Загрузка threads в Qdrant
# ---------------------------------------------------

def ingest_threads():
    df = load_all_joined_df()
    docs = build_thread_docs(df)

    if not docs:
        print("[threads] no docs to insert")
        return

    ids = make_thread_ids(docs)
    texts = [d.page_content for d in docs]
    metadatas = [d.metadata for d in docs]

    threads_qv.add_texts(
        texts=texts,
        metadatas=metadatas,
        ids=ids
    )

    print(f"[threads] DONE: docs_inserted_total={len(docs)}")




In [18]:
# ---------------------------------------------------
# 14. Запуск
# ---------------------------------------------------

# Сначала сообщения
ingest_messages(batch_size=1000)

# Потом треды
ingest_threads()

[messages] rows_processed=489, docs_inserted_total=2413, last_batch_rows=489, last_batch_docs=2413
[messages] DONE: rows_processed=489, docs_inserted_total=2413
[threads] DONE: docs_inserted_total=1073


### агент

In [23]:
import json
from collections import defaultdict
from langchain.tools import tool
from collections import defaultdict
from pathlib import Path

from langchain.tools import tool

SUMMARY_DIR = Path("summaries")
SUMMARY_DIR.mkdir(exist_ok=True)

In [24]:



@tool
def search_project_emails(project_hint: str, limit: int = 200) -> str:
    """
    Найти письма по проекту или теме через семантический поиск в коллекции mailkb_messages,
    затем сгруппировать найденные сообщения по thread_key.

    Аргументы:
        project_hint: Название проекта, код ЗНИ, тема, ключевые слова.
        limit: Максимальное количество найденных сообщений.

    Возвращает:
        JSON-строку вида:
        {
          "project_hint": "...",
          "threads": [
            {
              "thread_key": "...",
              "subject": "...",
              "participants": [...],
              "messages": [
                {
                  "email_id": "...",
                  "message_id": "...",
                  "sent_at_utc": "...",
                  "subject": "...",
                  "topic": "...",
                  "participants": [...],
                  "keywords": [...],
                  "folder": "...",
                  "snippet": "..."
                }
              ]
            }
          ]
        }
    """
    docs = messages_qv.similarity_search(project_hint, k=limit)

    threads = defaultdict(list)

    for doc in docs:
        md = doc.metadata or {}

        thread_key = md.get("thread_key") or "unknown_thread"
        sent_at = md.get("date") or md.get("sent_at_utc")

        message = {
            "email_id": md.get("email_id"),
            "message_id": md.get("message_id"),
            "sent_at_utc": sent_at,
            "subject": md.get("subject"),
            "topic": md.get("topic"),
            "participants": md.get("participants") or [],
            "keywords": md.get("keywords") or [],
            "folder": md.get("folder"),
            "snippet": (doc.page_content[:600] + "…") if doc.page_content else None,
        }

        threads[thread_key].append(message)

    thread_list = []

    for thread_key, messages in threads.items():
        messages_sorted = sorted(
            messages,
            key=lambda m: m.get("sent_at_utc") or "",
        )

        subject = None
        for m in messages_sorted:
            if m.get("subject"):
                subject = m["subject"]
                break

        participants_set = set()
        for m in messages_sorted:
            for p in m.get("participants") or []:
                participants_set.add(p)

        thread_list.append(
            {
                "thread_key": thread_key,
                "subject": subject,
                "participants": sorted(participants_set),
                "messages": messages_sorted,
            }
        )

    result = {
        "project_hint": project_hint,
        "threads": thread_list,
    }

    return json.dumps(result, ensure_ascii=False, indent=2)


@tool
def search_emails_raw(query: str, limit: int = 50) -> str:
    """
    Выполнить обычный семантический поиск по коллекции mailkb_messages
    без группировки по тредам.

    Аргументы:
        query: Произвольный текстовый запрос.
        limit: Максимальное количество результатов.

    Возвращает:
        JSON-строку вида:
        {
          "query": "...",
          "results": [...]
        }
    """
    docs = messages_qv.similarity_search(query, k=limit)

    data = []
    for doc in docs:
        md = doc.metadata or {}
        data.append({
            "thread_key": md.get("thread_key"),
            "email_id": md.get("email_id"),
            "message_id": md.get("message_id"),
            "subject": md.get("subject"),
            "topic": md.get("topic"),
            "sent_at_utc": md.get("date") or md.get("sent_at_utc"),
            "participants": md.get("participants") or [],
            "keywords": md.get("keywords") or [],
            "folder": md.get("folder"),
            "snippet": (doc.page_content[:400] + "…") if doc.page_content else None,
        })

    return json.dumps(
        {"query": query, "results": data},
        ensure_ascii=False,
        indent=2
    )


@tool
def search_project_emails_batch(project_hint: str, offset: int = 0, batch_size: int = 50) -> str:
    """
    Вернуть батч найденных сообщений по проекту или теме из коллекции mailkb_messages.

    Используется агентом для пошаговой пакетной обработки переписки.

    Аргументы:
        project_hint: Название проекта, код, тема или ключевые слова.
        offset: Смещение внутри выдачи.
        batch_size: Размер батча.

    Возвращает:
        JSON-строку вида:
        {
          "project_hint": "...",
          "offset": 0,
          "batch_size": 50,
          "batch_len": 50,
          "has_more": true,
          "batch": [...]
        }
    """
    docs = messages_qv.similarity_search(project_hint, k=offset + batch_size)
    docs = docs[offset: offset + batch_size]

    batch = []
    for d in docs:
        md = d.metadata or {}
        batch.append({
            "email_id": md.get("email_id"),
            "message_id": md.get("message_id"),
            "subject": md.get("subject"),
            "topic": md.get("topic"),
            "snippet": d.page_content[:600] if d.page_content else None,
            "sent_at_utc": md.get("date") or md.get("sent_at_utc"),
            "thread_key": md.get("thread_key"),
            "participants": md.get("participants") or [],
            "keywords": md.get("keywords") or [],
            "folder": md.get("folder"),
        })

    return json.dumps({
        "project_hint": project_hint,
        "offset": offset,
        "batch_size": batch_size,
        "batch_len": len(batch),
        "has_more": len(batch) == batch_size,
        "batch": batch
    }, ensure_ascii=False, indent=2)


@tool
def save_summary(summary_text: str, batch_id: int) -> str:
    """
    Сохранить summary текущего батча в файл summaries/summary_batch_{batch_id}.txt

    Аргументы:
        summary_text: Текст summary.
        batch_id: Порядковый номер батча.

    Возвращает:
        Строку с путём к сохранённому файлу.
    """
    p = SUMMARY_DIR / f"summary_batch_{batch_id}.txt"
    p.write_text(summary_text, encoding="utf-8")
    return f"saved_to={str(p)}"

In [25]:
SYSTEM_PROMPT = """
Ты — аналитик проектной переписки.

Твоя задача — обработать переписку по проекту порциями (батчами),
создать summary каждого батча и сохранить его через инструмент save_summary.

Итоговый глобальный отчёт по проекту делать НЕ нужно.

========================================
ОБЩИЙ АЛГОРИТМ
========================================

Шаг 1. Определи project_hint из запроса пользователя.
Это может быть:
- название проекта,
- код ЗНИ,
- тема,
- ключевые слова.

Шаг 2. Запусти цикл пакетной обработки:

  2.1. Вызови search_project_emails_batch(project_hint, offset, batch_size=50)

  2.2. Получи:
       - batch
       - offset
       - batch_size
       - batch_len
       - has_more

  2.3. На основе ТОЛЬКО этого batch сделай summary:
       - какие темы обсуждаются
       - какие вопросы поднимаются
       - какие решения или договорённости встречаются
       - какие проблемы, риски или открытые вопросы видны
       - без глобальных выводов по всему проекту

  2.4. Вызови save_summary(summary_text, batch_id)

  2.5. Если has_more = true:
       - увеличь offset на batch_size
       - перейди к следующему батчу

       Иначе:
       - закончи цикл
       - сообщи, что все батчи обработаны

========================================
ОГРАНИЧЕНИЯ
========================================
- НЕ делай глобальный отчёт по проекту
- НЕ объединяй выводы между разными батчами
- НЕ используй search_project_emails
- Основной инструмент поиска: search_project_emails_batch
- search_emails_raw можно использовать только как вспомогательный инструмент для точечной проверки

========================================
ФОРМАТ SUMMARY
========================================
Для каждого батча summary должен содержать:
1. Основные темы батча
2. Что обсуждалось
3. Найденные решения или договорённости
4. Проблемы / риски / открытые вопросы
5. Краткий вывод по батчу
"""

In [26]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    PIIMiddleware,
    SummarizationMiddleware,
)

agent = create_agent(
    model="deepseek-chat",
    tools=[
        search_project_emails_batch,
        save_summary,
        search_emails_raw,
    ],
    system_prompt=SYSTEM_PROMPT,
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware(
            "phone_number",
            detector=(
                r"(?:\+?\d{1,3}[\s.-]?)?"
                r"(?:\(?\d{2,4}\)?[\s.-]?)?"
                r"\d{3,4}[\s.-]?\d{4}"
            ),
            strategy="redact",
        ),
        SummarizationMiddleware(
            model="deepseek-chat",
            max_tokens_before_summary=500,
        ),
    ],
)

In [27]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """Начни обработку проекта Segezha батчами.
            Делай summary каждого батча и сохраняй их через save_summary.
            Итоговый отчёт НЕ делай. """

        }
    ]
})

report = result["messages"][-1].content
print(report)


KeyboardInterrupt: 